# Coffee quality benchmark – training pipeline

Standalone training pipeline derived from the production-ready code in
`productionReadyCode`. The pipeline predicts the continuous
`coffee_quality_score` target.

**Steps**

1. Configuration
2. Data loading and validation
3. Feature preparation (removal of leakage columns)
4. Train/test split
5. Preprocessing (median/mode imputation, scaling, one-hot encoding)
6. Training of the candidate models (RandomForest, XGBoost, LightGBM)
7. Evaluation and selection of the best model (test R²)
8. Persisting the artifacts to `pipelines/artifacts`
9. Verification of the saved artifacts

The saved artifacts are consumed by
`coffee-quality-pricing-benchmark_inference_pipeline.ipynb`.

In [1]:
"""Training pipeline for the coffee quality benchmark model."""

import json
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. Configuration

Paths are resolved relative to the `pipelines` folder, so the notebook
has to be run with `pipelines` as the working directory (default in VS
Code and Jupyter).

In [2]:
PIPELINE_DIR = Path.cwd()
PROJECT_ROOT = PIPELINE_DIR.parent
DATA_PATH = PROJECT_ROOT / "data" / "coffee_value_benchmark.csv"
ARTIFACTS_DIR = PIPELINE_DIR / "artifacts"

MODEL_FILE_NAME = "coffee_quality_model.joblib"
PREPROCESSOR_FILE_NAME = "preprocessor.joblib"
SUMMARY_FILE_NAME = "model_summary.json"

TARGET_COLUMN = "coffee_quality_score"
LEAKAGE_COLUMNS = (
    "record_id",
    "lot_id",
    "quality_grade",
    "price_usd_per_kg",
)
TEST_SIZE = 0.2
RANDOM_STATE = 42
SELECTION_METRIC = "r2"

print(f"Data path:     {DATA_PATH}")
print(f"Artifacts dir: {ARTIFACTS_DIR}")

Data path:     c:\D\Work\Ismeret\AI_ML_DL_TL\CUBIX_MLOPS_AIOPS\coffeePricePredictor\data\coffee_value_benchmark.csv
Artifacts dir: c:\D\Work\Ismeret\AI_ML_DL_TL\CUBIX_MLOPS_AIOPS\coffeePricePredictor\pipelines\artifacts


## 2. Data loading and validation

In [3]:
def load_dataset(data_path: Path) -> pd.DataFrame:
    """Load the benchmark CSV file and validate that it is not empty."""
    if not data_path.exists():
        raise FileNotFoundError(f"Dataset not found: {data_path}")

    dataframe = pd.read_csv(data_path)
    if dataframe.empty:
        raise ValueError(f"Dataset is empty: {data_path}")
    return dataframe


def validate_dataset(
    dataframe: pd.DataFrame,
    target_column: str,
    leakage_columns: tuple[str, ...],
) -> None:
    """Check that the required columns and target values are present."""
    required_columns = {target_column, *leakage_columns}
    missing = required_columns - set(dataframe.columns)
    if missing:
        raise ValueError(
            f"Dataset is missing required columns: {sorted(missing)}"
        )
    if dataframe[target_column].isna().any():
        raise ValueError(
            f"Target column '{target_column}' contains missing values."
        )

In [4]:
raw_data = load_dataset(DATA_PATH)
validate_dataset(raw_data, TARGET_COLUMN, LEAKAGE_COLUMNS)

print(f"Dataset shape: {raw_data.shape}")
raw_data.head()

Dataset shape: (35700, 36)


,record_id,lot_id,harvest_season,harvest_calendar_year,harvest_month,country_of_origin,farm_type,farm_size_hectares,tree_age_years,altitude_mean_meters,...,lot_size_kg,roast_level,acidity_score,sweetness_score,body_score,aroma_score,flavor_score,coffee_quality_score,quality_grade,price_usd_per_kg
0,REC-0000001,LOT-0032936,2020/21,2020,10,Vietnam,Smallholder,0.58,28.0,1961.0,...,26433.0,Medium,7.95,7.30,8.45,7.35,7.37,78.01,Very Good,2.43
1,REC-0000002,LOT-0030952,2021/22,2022,1,Honduras,Smallholder,1.12,2.0,1257.0,...,25446.0,Medium,7.62,8.17,7.45,7.80,6.88,75.62,Good,4.06
2,REC-0000003,LOT-0018572,2020/21,2020,11,Vietnam,Cooperative,13.82,5.0,557.0,...,15707.0,Medium,8.47,8.83,8.13,7.70,8.10,72.41,Commercial,4.26
3,REC-0000004,LOT-0005318,2021/22,2021,10,Vietnam,Cooperative,9.96,22.0,652.0,...,2879.0,Medium,7.54,8.12,8.21,7.77,7.13,75.67,Good,3.60
4,REC-0000005,LOT-0006317,2021/22,2021,7,Brazil,Smallholder,1.90,11.0,971.0,...,9561.0,Medium,6.70,8.08,8.68,7.38,7.16,73.77,Good,4.16


In [5]:
missing_values = raw_data.isna().sum()
missing_values[missing_values > 0].sort_values(ascending=False)

certification             21723
humidity_pct               2282
avg_annual_rainfall_mm     1819
bean_density_g_ml          1632
moisture_pct               1612
dtype: int64

## 3. Feature preparation

Identifier and outcome columns (`record_id`, `lot_id`, `quality_grade`,
`price_usd_per_kg`) are removed to avoid target leakage.

In [6]:
def prepare_features(
    dataframe: pd.DataFrame,
    target_column: str,
    leakage_columns: tuple[str, ...],
) -> tuple[pd.DataFrame, pd.Series]:
    """Split the data into features and target without leakage columns."""
    drop_columns = [*leakage_columns, target_column]
    features = dataframe.drop(columns=drop_columns, errors="ignore")
    target = dataframe[target_column]
    return features, target


features, target = prepare_features(raw_data, TARGET_COLUMN, LEAKAGE_COLUMNS)

print(f"Number of features: {features.shape[1]}")
target.describe()

Number of features: 31


count    35700.000000
mean        77.602334
std          4.915255
min         60.000000
25%         73.810000
50%         78.240000
75%         81.330000
max         94.180000
Name: coffee_quality_score, dtype: float64

## 4. Train/test split

In [7]:
x_train, x_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)

print(f"Train rows: {len(x_train)}")
print(f"Test rows:  {len(x_test)}")

Train rows: 28560
Test rows:  7140


## 5. Preprocessing

* numeric columns: median imputation + standard scaling
* categorical columns: most frequent imputation + one-hot encoding

The preprocessor is fitted on the training split only.

In [8]:
def build_preprocessor(features: pd.DataFrame) -> ColumnTransformer:
    """Create a preprocessing pipeline for numeric and categorical data."""
    numerical_columns = features.select_dtypes(
        include=["number"]
    ).columns.tolist()
    categorical_columns = features.select_dtypes(
        exclude=["number"]
    ).columns.tolist()
    if not numerical_columns:
        raise ValueError("No numeric columns were found in the feature set.")

    numerical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "encoder",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numerical_pipeline, numerical_columns),
            ("cat", categorical_pipeline, categorical_columns),
        ]
    )


preprocessor = build_preprocessor(x_train)
x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)

print(f"Processed train shape: {x_train_processed.shape}")
print(f"Processed test shape:  {x_test_processed.shape}")

Processed train shape: (28560, 61)
Processed test shape:  (7140, 61)


## 6. Training of the candidate models

In [9]:
def build_models(random_state: int) -> dict[str, Any]:
    """Define the candidate regression models of the benchmark study."""
    return {
        "RandomForest": RandomForestRegressor(
            n_estimators=200,
            max_depth=15,
            n_jobs=-1,
            random_state=random_state,
        ),
        "XGBoost": xgb.XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            n_jobs=-1,
            random_state=random_state,
            verbosity=0,
        ),
        "LightGBM": lgb.LGBMRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            n_jobs=-1,
            random_state=random_state,
            verbose=-1,
        ),
    }


def evaluate_predictions(
    y_true: pd.Series,
    y_pred: np.ndarray,
) -> dict[str, float]:
    """Compute regression metrics for a prediction set."""
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def train_and_evaluate(
    models: dict[str, Any],
    x_train: np.ndarray,
    y_train: pd.Series,
    x_test: np.ndarray,
    y_test: pd.Series,
) -> tuple[dict[str, dict[str, Any]], dict[str, Any]]:
    """Fit every candidate model and evaluate it on both splits."""
    results: dict[str, dict[str, Any]] = {}
    trained_models: dict[str, Any] = {}
    for model_name, model in models.items():
        start = time.perf_counter()
        model.fit(x_train, y_train)
        elapsed = time.perf_counter() - start

        results[model_name] = {
            "train": evaluate_predictions(y_train, model.predict(x_train)),
            "test": evaluate_predictions(y_test, model.predict(x_test)),
            "time_seconds": round(elapsed, 3),
        }
        trained_models[model_name] = model
        print(f"{model_name} trained in {elapsed:.2f} s")
    return results, trained_models

In [10]:
results, trained_models = train_and_evaluate(
    build_models(RANDOM_STATE),
    x_train_processed,
    y_train,
    x_test_processed,
    y_test,
)

RandomForest trained in 18.98 s


XGBoost trained in 1.38 s


LightGBM trained in 0.85 s


## 7. Evaluation and selection of the best model

In [11]:
comparison = pd.DataFrame(
    {
        model_name: {
            f"{split}_{metric}": result[split][metric]
            for split in ("train", "test")
            for metric in ("rmse", "mae", "r2")
        }
        | {"time_seconds": result["time_seconds"]}
        for model_name, result in results.items()
    }
).T.sort_values(f"test_{SELECTION_METRIC}", ascending=False)

comparison

,train_rmse,train_mae,train_r2,test_rmse,test_mae,test_r2,time_seconds
LightGBM,1.475656,1.173199,0.909844,1.670481,1.323586,0.884598,0.853
XGBoost,1.385374,1.097875,0.920538,1.674202,1.324503,0.884083,1.382
RandomForest,0.888135,0.710752,0.967343,1.740039,1.374860,0.874787,18.975


In [12]:
best_model_name = max(
    results,
    key=lambda name: results[name]["test"][SELECTION_METRIC],
)
best_model = trained_models[best_model_name]

print(f"Best model: {best_model_name}")
print(json.dumps(results[best_model_name], indent=2))

Best model: LightGBM
{
  "train": {
    "rmse": 1.475655992497352,
    "mae": 1.1731986330856086,
    "r2": 0.9098440033244417
  },
  "test": {
    "rmse": 1.6704806257554707,
    "mae": 1.3235855525591058,
    "r2": 0.884597760318714
  },
  "time_seconds": 0.853
}


## 8. Persisting the artifacts

* `coffee_quality_model.joblib` – selected best model
* `preprocessor.joblib` – fitted preprocessing transformer
* `model_summary.json` – metrics of all models and training metadata

In [13]:
def build_summary(
    best_model_name: str,
    results: dict[str, dict[str, Any]],
    feature_columns: list[str],
    train_rows: int,
    test_rows: int,
) -> dict[str, Any]:
    """Collect model metrics and training metadata."""
    return {
        "best_model": best_model_name,
        "selection_metric": SELECTION_METRIC,
        "best_model_metrics": results[best_model_name],
        "all_models": results,
        "target_column": TARGET_COLUMN,
        "feature_columns": feature_columns,
        "leakage_columns": list(LEAKAGE_COLUMNS),
        "data_path": str(DATA_PATH),
        "train_rows": train_rows,
        "test_rows": test_rows,
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "trained_at": datetime.now(timezone.utc).isoformat(),
        "library_versions": {
            "pandas": pd.__version__,
            "numpy": np.__version__,
            "scikit-learn": sklearn.__version__,
            "xgboost": xgb.__version__,
            "lightgbm": lgb.__version__,
            "joblib": joblib.__version__,
        },
    }


def save_artifacts(
    output_dir: Path,
    model: Any,
    preprocessor: ColumnTransformer,
    summary: dict[str, Any],
) -> dict[str, Path]:
    """Persist the model, the preprocessor and the summary to disk."""
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = {
        "model": output_dir / MODEL_FILE_NAME,
        "preprocessor": output_dir / PREPROCESSOR_FILE_NAME,
        "summary": output_dir / SUMMARY_FILE_NAME,
    }
    joblib.dump(model, paths["model"])
    joblib.dump(preprocessor, paths["preprocessor"])
    paths["summary"].write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8",
    )
    return paths

In [14]:
summary = build_summary(
    best_model_name=best_model_name,
    results=results,
    feature_columns=features.columns.tolist(),
    train_rows=len(x_train),
    test_rows=len(x_test),
)
artifact_paths = save_artifacts(
    ARTIFACTS_DIR,
    best_model,
    preprocessor,
    summary,
)

for name, path in artifact_paths.items():
    print(f"{name:<13}: {path}")

model        : c:\D\Work\Ismeret\AI_ML_DL_TL\CUBIX_MLOPS_AIOPS\coffeePricePredictor\pipelines\artifacts\coffee_quality_model.joblib
preprocessor : c:\D\Work\Ismeret\AI_ML_DL_TL\CUBIX_MLOPS_AIOPS\coffeePricePredictor\pipelines\artifacts\preprocessor.joblib
summary      : c:\D\Work\Ismeret\AI_ML_DL_TL\CUBIX_MLOPS_AIOPS\coffeePricePredictor\pipelines\artifacts\model_summary.json


## 9. Verification of the saved artifacts

The artifacts are reloaded from disk and the test metrics are recomputed
to make sure the persisted model reproduces the evaluated results.

In [15]:
loaded_model = joblib.load(artifact_paths["model"])
loaded_preprocessor = joblib.load(artifact_paths["preprocessor"])

reloaded_metrics = evaluate_predictions(
    y_test,
    loaded_model.predict(loaded_preprocessor.transform(x_test)),
)
expected_metrics = results[best_model_name]["test"]
for metric, value in reloaded_metrics.items():
    if not np.isclose(value, expected_metrics[metric]):
        raise AssertionError(f"Metric mismatch after reload: {metric}")

print("Artifacts verified successfully.")
print(json.dumps(reloaded_metrics, indent=2))

Artifacts verified successfully.


{
  "rmse": 1.6704806257554707,
  "mae": 1.3235855525591058,
  "r2": 0.884597760318714
}
